# Coroner: smoke test on a free Colab T4

Goal: prove the training pipeline works end to end on a **small** model (about 20 to 30 minutes). Accuracy will be low after 60 steps. What we check:
1. training runs without errors and the loss goes down,
2. the model replies with JSON that parses,
3. the scorer reads the replies.

Runtime: **Runtime, Change runtime type, T4 GPU**. Run the cells from top to bottom. If a cell fails, copy the red error text and send it back.

In [ ]:
!nvidia-smi

## 1. Install (about 3 minutes)

In [ ]:
%%capture
!pip install unsloth

## 2. Get the code and the small dataset

In [ ]:
!git clone https://github.com/coroner-hash/ai.git
%cd ai
!ls data/smoke

## 3. Train (60 steps)
Look for: `600 examples`, a token count under 2048, and `loss` printed every 5 steps that trends down.

In [ ]:
import os
os.environ.update(
    MODEL="unsloth/Qwen3.5-0.8B",
    MAX_STEPS="60",
    TRAIN_FILE="data/smoke/train.jsonl",
    OUT_DIR="outputs/smoke",
)
!python training/train_lora.py

## 4. Generate replies on the 90 eval rows (about 5 to 10 minutes)

In [ ]:
!python training/eval_local.py --adapter outputs/smoke --eval data/smoke/eval.jsonl --out outputs/replies.jsonl

## 5. Score them
With 60 steps expect `json_valid` well above 0.5 and low accuracy. That is fine for a smoke test.

In [ ]:
!python training/eval_model.py --replies outputs/replies.jsonl --eval data/smoke/eval.jsonl

## 6. Look at three replies yourself

In [ ]:
import json
rows = [json.loads(l) for l in open("data/smoke/eval.jsonl")]
reps = [json.loads(l)["reply"] for l in open("outputs/replies.jsonl")]
for i in (0, 31, 61):
    print("MODE:", rows[i]["meta"]["mode"], rows[i]["meta"]["source"])
    print("WANT:", rows[i]["messages"][2]["content"][:500])
    print("GOT :", reps[i][:500])
    print("-" * 60)

## What to send back
- The training log lines with `loss` (first and last), or the error text if it stopped.
- The table from step 5.
- Anything odd in step 6 (empty replies, text instead of JSON, repeated text).